## Import necessary libraries

In [1]:
import numpy as np
import random
import tensorflow as tf
from tensorflow.keras import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, LeakyReLU, BatchNormalization
from tensorflow.keras.callbacks import LearningRateScheduler, ReduceLROnPlateau, EarlyStopping
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## load the data

In [ ]:
combined_df = pd.read_csv("combined_features_pre.csv")

In [ ]:
features_df= combined_df.drop(columns=['Trajectory', 'Frame'])
numeric_df = features_df
numeric_df

## Standardize the data

In [ ]:
scaler = MinMaxScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_train_scaled = pd.DataFrame(x_train_scaled, columns=x_train.columns) 
x_train_scaled

## Autoencoder

In [ ]:
# Set random seeds for reproducibility
random.seed(62)
np.random.seed(62)
tf.random.set_seed(62)

# Autoencoder Model
class AutoEncoders(Model):
    def __init__(self, output_units):
        super().__init__()
        self.encoder = Sequential([
            Dense(8, kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
            BatchNormalization(),
            LeakyReLU(alpha=0.2),
            Dropout(0.3),
            Dense(4),
            BatchNormalization(),
            LeakyReLU(alpha=0.2),
            Dense(2) 
        ])
        self.decoder = Sequential([
            Dense(4),
            BatchNormalization(),
            LeakyReLU(alpha=0.2),
            Dense(8),
            BatchNormalization(),
            LeakyReLU(alpha=0.2),
            Dense(output_units, activation="sigmoid")
        ])
    
    def call(self, inputs):
        encoded = self.encoder(inputs)
        decoded = self.decoder(encoded)
        return decoded

# Learning rate scheduler function
def lr_scheduler(epoch, lr):
    if epoch > 10:
        return lr * 0.5  

# Callbacks
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
early_stopping = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

# Create and compile the autoencoder
auto_encoder = AutoEncoders(len(x_train_scaled[0]))
auto_encoder.compile(
    loss='mae', 
    metrics=['mae', 'mse'],
    optimizer=tf.keras.optimizers.Nadam(learning_rate=1e-3)
)

# Train autoencoder
history = auto_encoder.fit(
    x_train_scaled,
    x_train_scaled,
    epochs=100,
    batch_size=256,
    validation_split=0.2,
    shuffle=True,
    callbacks=[LearningRateScheduler(lr_scheduler), reduce_lr, early_stopping]
)

# latent space representation
reduced_data = auto_encoder.encoder.predict(x_train_scaled)

# Check for NaN values in the latent space
if np.any(np.isnan(reduced_data)):
    print("Latent space contains NaN values.")
else:
    print("No NaN values in the latent space.")

# Reduced data 
reduced_df = pd.DataFrame(reduced_data, columns=[f'latent_{i}' for i in range(reduced_data.shape[1])])

# Latent Space Representation
plt.figure(figsize=(8, 6))
plt.scatter(reduced_data[:, 0], reduced_data[:, 1], alpha=0.5)
plt.title("Latent Space Representation")
plt.xlabel("Latent Dimension 1")
plt.ylabel("Latent Dimension 2")
plt.grid()
plt.show()

## Feature Contribution Analysis (using Correlation)

In [ ]:
correlations = pd.concat([pd.DataFrame(x_train_scaled), reduced_df], axis=1).corr()
feature_contributions = correlations.loc[:len(x_train_scaled[0])-1, reduced_df.columns]

# Print Feature Contributions
print("Feature Contributions:\n", feature_contributions)

# Visualization: Feature Contribution Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(feature_contributions, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Feature Contributions to Latent Dimensions")
plt.show()

## Loss Curves

In [ ]:
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Loss Curves')
plt.legend()
plt.show()

## Free Energy Surface 

In [ ]:
def compute_free_energy(data, bins=3000, temp=298):
    kB = 0.001987  # Boltzmann constant in kcal/(mol·K)
    hist, xedges, yedges = np.histogram2d(data[:, 0], data[:, 1], bins=bins)
    
    # Convert to probabilities
    prob = hist / np.sum(hist)
    
    # Calculate free energy in kcal/mol
    with np.errstate(divide='ignore'):
        free_energy = -kB * temp * np.log(prob)
    
    # Normalize: Set infinite or NaN values to max free energy
    free_energy = free_energy - np.nanmin(free_energy)  
    free_energy[np.isinf(free_energy)] = np.nanmax(free_energy)
    
    return free_energy, xedges, yedges

# Generate free energy surface
bins = 180 
free_energy, xedges, yedges = compute_free_energy(reduced_data, bins=bins)

# Plot Free Energy Surface
plt.figure(figsize=(10, 8))
extent = [xedges[0], xedges[-1], yedges[0], yedges[-1]]
plt.imshow(free_energy.T, origin='lower', extent=extent, cmap='plasma', aspect='auto')

cbar = plt.colorbar()
cbar.set_label("Free Energy (kcal/mol)", fontsize=14) 
plt.xlabel("Latent 0", fontsize=16)  
plt.ylabel("Latent 1", fontsize=16)  
plt.title("Free Energy Surface", fontsize=16) 
plt.xticks(fontsize=12) 
plt.yticks(fontsize=12)
plt.show()
